# Problem:

At Arty's Tire Repair in Campbell, they would like to reachout to customers who have had their oil changes done three months ago to remind them they are likely due for another. However, with their current system layout with ALLDATA they would have to select a range of dates and manually parse all invoices to see what customer had an oil change. As things stand now, much time is spent simply parsing through invoices and they would like to minimize it.

# Solution:

I've come in to automate the process of parsing for customer's oil change job status and contact information by utitilzing ALLDATA's clump invoice file for the specified date range and python to provide the name, phone number, and vehicle for all customers who have had their oil change done for the specified dates.

In [1]:
# importing required modules
from pypdf import PdfReader
import re

In [6]:
def extract_page_index(text):
    """
    The ALLDATA invoice layout of each invoice has a footer detailed "[what page] of [how many]".
    We'll utilize this to reference what customer we are refering to in the case that our customer information
    is not on the same page as the oil change information. This will also serve to allow as a measure of
    how many pages a customer has left on an already scraped invoice-- so that we may save time and computational 
    resources by not even checking the rest of the invoice.
    
    Parameter(s):
     text - a string of text found in a focused page from the imported pdf
    
    Return(s):
     curr_page -- an int refering to what page of an invoice we are currently reading.
     Will return None if no such information is found 
     
     total_pages -- an int refering the the total number of pages of the current invoice we are reading.
     Will return None if no such information is found 
    
    """
    # Look for patterns like "Page 1 of 2", "Page 10 of 15", etc.
    match = re.search(r'Page\s+(\d+)\s+of\s+(\d+)', text, re.IGNORECASE)
    if match:
        current_page = int(match.group(1))
        total_pages = int(match.group(2))
        return current_page, total_pages
    
    print("Error: Could not detect page index")
    return None, None




def contains_synthetic_oil_change(text):
    """
    Before any customer information scraping, we want to see if the page being examined contains text signifying
    that an oil change was done.
    
    Parameter(s):
     text - a string of text found in a focused page from the imported pdf
    
    Return(s):
     boolean representing whether or not the text "synthetic oil change" is found in the page text
    
    """
    return "synthetic oil change" in text.lower()

def extract_name_and_phone(text):
    """
    This function and the rest below will only run if oil change information is found on this invoice.
    This function will extract the customer name and phone number.
    
    
    Parameter(s):
     text - a string of text found in a focused page from the imported pdf
    
    Return(s):
     name -- a string of the name of the focused invoice.
     Will return None if no such information is found 
     
     phone -- a string of phone number belonging to the focused invoice's customer.
     Will return None if no such information is found 
     
    """
    lines = text.splitlines()
    email_pattern = re.compile(r'.*artysautorepair@gmail\.com', re.IGNORECASE)
    digit_pattern = re.compile(r'\d')
    name_count = 0
    name = None
    phone = None
    for i, line in enumerate(lines):
        if email_pattern.search(line):
            
            if email_pattern.search(line):
                if name_count > 1:
                    print("Error: More than one name found on a page")
                    return name, phone
                
            # Get next non-empty line = name
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j >= len(lines):
                continue
            name = lines[j].strip()

            # Get the line after the name
            k = j + 1
            while k < len(lines) and not lines[k].strip():
                k += 1
            if k >= len(lines):
                continue
            next_line = lines[k].strip()

            # Extract first 10 digits from that line
            digits_found = digit_pattern.findall(next_line)
            phone = ''.join(digits_found[:10])

    return name, phone

def extract_vehicle_info(text):
    """
    If a oil change information was found in this invoice then this function is called to extract
    the vehicle information text.
    
    Parameter(s):
     text - a string of text found in a focused page from the imported pdf
    
    Return(s):
     vehicle  -- a string of the vehicle of the focused invoice.
     Will return None if no such information is found 
     
    """
    
    # The pattern below matches:
    # - Year (4 digits starting with 19 or 20)
    # - 1 to 4 capitalized tokens (words with optional digits/hyphens)
    # - Followed by "Miles In:" anywhere later on the same line
    pattern = re.compile(
        r'(\d{4}(?:\s+[A-Z][a-zA-Z0-9\-]*){1,4})\s+Miles In:',
        re.IGNORECASE
    )

    vehicles = None
    for match in pattern.finditer(text):
        vehicle = match.group(1).strip()

    return vehicle






In [5]:
def main():
    # creating a pdf reader object
    path = "C:/Users/arias/OneDrive/Documents/Arty's/Jan Invoices.pdf"
    reader = PdfReader(path)
    oil_change_found = False
    first_page = None
    for index, page in enumerate(reader.pages):
        text = page.extract_text()
    
        curr_page_index, num_of_total_pages = extract_page_index(text)
    
        if curr_page_index == 1:
            first_page = page
    
        if contains_synthetic_oil_change(text) and not oil_change_found:
            oil_change_found = True
            if curr_page_index != 1:
                text = first_page.extract_text()
            
            name, phone = extract_name_and_phone(text)
            vehicle = extract_vehicle_info(text)
        
            if vehicle == None:
                print(text)
            print(f"{name:<35} Phone#: ({phone[0:3]}){phone[3:6]}-{phone[6:10]}   Vehicle: {vehicle:<30}")
        
        if curr_page_index == num_of_total_pages:
            oil_change_found = False
        
if __name__ == "__main__":
    main()


Hector Nava (NAV001)                Phone#: (408)701-7218   Vehicle: 2019 Nissan Sentra SR         
Eric & Destiny Hernanadez (HER003)  Phone#: (619)727-7353   Vehicle: 2004 Subaru Forester XS       
Bruno Skracic (SKR001)              Phone#: (650)417-8594   Vehicle: 2009 Toyota Corolla LE        
Angel Quispe (QUI001)               Phone#: (408)624-2733   Vehicle: 2019 Honda Odyssey EX-L       
Angel Caceres (CAC001)              Phone#: (408)991-2969   Vehicle: 2012 Nissan Rogue S           
Nguyen Tran (TRA002)                Phone#: (925)366-1036   Vehicle: 2021 Toyota Prius Prime XLE   
Dale & Deborah Velasquez (VEL001)   Phone#: (650)564-9062   Vehicle: 2020 Ford Explorer XLT        
Milan Adhikari (ADH001)             Phone#: (214)796-3858   Vehicle: 2019 Toyota Prius LE          
Scott Jenning (JEN001)              Phone#: (408)398-8570   Vehicle: 2001 Toyota Avalon XLS        
Eric & Destiny Hernanadez (HER003)  Phone#: (619)727-7353   Vehicle: 2004 Toyota Camry LE          
